# XGBoost on Descriptors (Representation A)


## Step 1 - Prepare the data for XGBoost

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [7]:
features_df = pd.read_csv("training_features.csv")
features_df.head()


,Unnamed: 0,INDEX,SMILES,ACTIVE,MolFromSmiles,NumAtoms,NumHeavyAtoms,NumBonds,fr_Al_COO,fr_Al_OH,...,CalcNumRings,CalcNumRotatableBonds,CalcNumSaturatedCarbocycles,CalcNumSaturatedHeterocycles,CalcNumSaturatedRings,CalcNumSpiroAtoms,CalcNumUnspecifiedAtomStereoCenters,CalcPhi,CalcTPSA,_CalcMolWt
0,0,1,O=C(Nc1ccc2c(c1)OCCO2)C1CCN(c2ncccn2)CC1,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0040>,25,25,28,0,0,...,4,3,0,1,1,0,0,4.368063,76.58,340.383
1,1,2,COCCCN1C(=O)C2C(C(=O)Nc3cccc(Cl)c3)C3C=CC2(O3)...,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea00b0>,35,35,39,0,0,...,5,8,1,2,3,1,5,6.877647,96.97,502.011
2,2,3,CCSc1ncc(Cl)c(C(=O)Nc2ccccc2C)n1,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0120>,20,20,21,0,0,...,2,4,0,0,0,0,0,4.977937,54.88,307.806
3,3,4,COc1ccc2cc(/C=N/NC(=O)CN(c3ccccc3C)S(=O)(=O)c3...,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0190>,36,36,39,0,0,...,4,8,0,0,0,0,0,7.516779,100.96,523.014
4,4,5,CCCC(=O)Nc1nc2ccc(NC(=O)c3c(F)c(F)c(OC)c(F)c3F...,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0200>,30,30,32,0,0,...,3,6,0,0,0,0,0,6.202841,80.32,441.406


## Step 2 - Split into X (features) and y (targets)

In [8]:
features_df.columns

Index(['Unnamed: 0', 'INDEX', 'SMILES', 'ACTIVE', 'MolFromSmiles', 'NumAtoms',
       'NumHeavyAtoms', 'NumBonds', 'fr_Al_COO', 'fr_Al_OH',
       ...
       'CalcNumRings', 'CalcNumRotatableBonds', 'CalcNumSaturatedCarbocycles',
       'CalcNumSaturatedHeterocycles', 'CalcNumSaturatedRings',
       'CalcNumSpiroAtoms', 'CalcNumUnspecifiedAtomStereoCenters', 'CalcPhi',
       'CalcTPSA', '_CalcMolWt'],
      dtype='object', length=157)

In [10]:
TARGET_COL = "ACTIVE"  
y = features_df[TARGET_COL]

In [15]:
non_feature_cols = [TARGET_COL, "SMILES", "MolFromSmiles", "Unnamed: 0", "INDEX"]
non_feature_cols = [c for c in non_feature_cols if c in features_df.columns]


y = features_df[TARGET_COL]

X = features_df.drop(columns=non_feature_cols)
X = X.select_dtypes(include=[np.number])

X.shape, y.shape

((202895, 152), (202895,))

## Step 3 - Handle missing values

In [16]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

## Step 4 - Train

In [17]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1  # usa tutti i core dentro XGBoost
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_scores = cross_val_score(
    xgb_clf,
    X_imputed,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1   # QUI la modifica, niente multiprocess sklearn
)

print("AUC per fold:", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Std AUC:", auc_scores.std())


AUC per fold: [0.89148221 0.89551735 0.89823882 0.88476005 0.88564934]
Mean AUC: 0.8911295553188812
Std AUC: 0.005301359368940537


## Step 5 - Tuning

In [19]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_dist = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1   # XGBoost usa tutti i core, ok
)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    n_jobs=1,          # QUI è la differenza importante
    random_state=42
)

random_search.fit(X_imputed, y)

print("Best AUC:", random_search.best_score_)
print("Best params:", random_search.best_params_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best AUC: 0.8999764005841053
Best params: {'subsample': 0.8, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.6}
